# Enhanced Model Evaluation for Trading

This notebook demonstrates how to use enhanced evaluation capabilities to comprehensively assess trading model performance.

## Setup

First, let's import the necessary modules and set up our environment.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from src.evaluation.enhanced_evaluator import EnhancedEvaluator
from src.data.binance_fetcher import BinanceFetcher
from src.models.base_model import BaseModel

## Data Preparation

Fetch historical data and prepare for evaluation.

In [ ]:
# Load configuration
with open('configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize data fetcher
fetcher = BinanceFetcher(config['data']['binance'])

# Fetch data
data = await fetcher.fetch_historical_data()

# Calculate returns
returns = data['close'].pct_change().dropna()
prices = data['close']

# Fetch benchmark data (e.g., BTC/USD)
benchmark_data = await fetcher.fetch_historical_data(symbol='BTCUSDT')
benchmark_returns = benchmark_data['close'].pct_change().dropna()

print(f"Data points: {len(returns)}")
print(f"Date range: {returns.index[0]} to {returns.index[-1]}")

## Model Predictions

Generate model predictions for evaluation.

In [ ]:
# Initialize model
model = BaseModel(config)

# Generate predictions
with torch.no_grad():
    predictions = model(torch.FloatTensor(data['features']))
    predictions = predictions.numpy()

# Convert to positions
positions = np.sign(predictions)

## Enhanced Evaluation

Perform comprehensive model evaluation.

In [ ]:
# Initialize evaluator
evaluator = EnhancedEvaluator(
    config=config,
    risk_free_rate=0.0,
    benchmark_returns=benchmark_returns.values
)

# Evaluate model
performance = evaluator.evaluate(
    predictions=predictions,
    targets=returns.values,
    prices=prices.values
)

# Print performance metrics
print("Performance Metrics:")
for metric, value in performance.metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nTrade Analysis:")
for metric, value in performance.trade_analysis.items():
    print(f"{metric}: {value:.4f}")

print("\nRisk Metrics:")
for metric, value in performance.risk_metrics.items():
    print(f"{metric}: {value:.4f}")

## Performance Visualization

Create comprehensive performance visualizations.

In [ ]:
# Plot evaluation results
evaluator.plot_results(performance)

# Additional analysis plots
plt.figure(figsize=(15, 10))

# Rolling Sharpe ratio
plt.subplot(2, 2, 1)
performance.rolling_metrics['sharpe_ratio'].plot()
plt.title('Rolling Sharpe Ratio')
plt.axhline(y=0, color='r', linestyle='--')

# Position distribution
plt.subplot(2, 2, 2)
sns.histplot(performance.positions, bins=50)
plt.title('Position Distribution')

# Trade return distribution
plt.subplot(2, 2, 3)
trade_returns = performance.returns[np.diff(performance.positions, prepend=0) != 0]
sns.histplot(trade_returns, bins=50)
plt.title('Trade Return Distribution')

# Rolling correlation with benchmark
plt.subplot(2, 2, 4)
correlation = pd.Series(performance.returns).rolling(252).corr(
    pd.Series(benchmark_returns[:len(performance.returns)])
)
correlation.plot()
plt.title('Rolling Correlation with Benchmark')

plt.tight_layout()
plt.show()

## Trade Analysis

Analyze individual trades and trading patterns.

In [ ]:
def analyze_trade_patterns(performance):
    """Analyze trading patterns."""
    trades = np.diff(performance.positions, prepend=0)
    trade_returns = performance.returns[trades != 0]
    trade_dates = returns.index[trades != 0]
    
    # Create trade DataFrame
    trade_df = pd.DataFrame({
        'date': trade_dates,
        'return': trade_returns,
        'direction': np.sign(trades[trades != 0])
    })
    
    # Add time features
    trade_df['hour'] = trade_df['date'].dt.hour
    trade_df['day_of_week'] = trade_df['date'].dt.dayofweek
    
    return trade_df

# Analyze trades
trade_df = analyze_trade_patterns(performance)

# Plot trade patterns
plt.figure(figsize=(15, 5))

# Returns by hour
plt.subplot(1, 3, 1)
trade_df.groupby('hour')['return'].mean().plot(kind='bar')
plt.title('Average Returns by Hour')
plt.xticks(rotation=45)

# Returns by day of week
plt.subplot(1, 3, 2)
trade_df.groupby('day_of_week')['return'].mean().plot(kind='bar')
plt.title('Average Returns by Day of Week')
plt.xticks(rotation=45)

# Trade direction distribution
plt.subplot(1, 3, 3)
trade_df['direction'].value_counts().plot(kind='bar')
plt.title('Trade Direction Distribution')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Risk Analysis

Perform detailed risk analysis.

In [ ]:
def analyze_risk_exposures(performance):
    """Analyze risk exposures."""
    # Calculate rolling volatility
    rolling_vol = pd.Series(performance.returns).rolling(20).std() * np.sqrt(252)
    
    # Calculate rolling VaR
    rolling_var = pd.Series(performance.returns).rolling(100).quantile(0.05)
    
    # Calculate rolling beta
    rolling_beta = pd.Series(performance.returns).rolling(252).cov(
        pd.Series(benchmark_returns[:len(performance.returns)])
    ) / pd.Series(benchmark_returns[:len(performance.returns)]).rolling(252).var()
    
    return rolling_vol, rolling_var, rolling_beta

# Analyze risk
rolling_vol, rolling_var, rolling_beta = analyze_risk_exposures(performance)

# Plot risk metrics
plt.figure(figsize=(15, 5))

# Rolling volatility
plt.subplot(1, 3, 1)
rolling_vol.plot()
plt.title('Rolling Volatility')

# Rolling VaR
plt.subplot(1, 3, 2)
rolling_var.plot()
plt.title('Rolling VaR (5%)')

# Rolling beta
plt.subplot(1, 3, 3)
rolling_beta.plot()
plt.title('Rolling Beta')
plt.axhline(y=1, color='r', linestyle='--')

plt.tight_layout()
plt.show()

## Performance Report

Generate comprehensive performance report.

In [ ]:
# Generate report
report = evaluator.generate_report(performance)
print(report)

# Save report and plots
os.makedirs('reports', exist_ok=True)
evaluator.generate_report(performance, 'reports/performance_report.txt')
evaluator.plot_results(performance, 'reports/performance_plots.png')

print("\nReport and plots saved to 'reports' directory")